In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
from langchain_community.document_loaders import PyPDFLoader

PDF_PATH = "echap01.pdf"  # adjust if needed

loader = PyPDFLoader(PDF_PATH)
pages = loader.load()

print(f"Loaded {len(pages)} pages from the PDF.")
print("\n--- First page preview (first 500 chars) ---")
print(pages[0].page_content[:500])

Loaded 35 pages from the PDF.

--- First page preview (first 500 chars) ---
11
CHAPTER
STATE OF THE ECONOMY: 
PUSHING THE GROWTH 
FRONTIER
The global economic environment remains uncertain, shaped by geopolitical 
tensions, trade disruptions, and divergent growth and inflation outcomes 
across major economies. While global activity has shown resilience in the near 
term, underlying vulnerabilities persist, including elevated fiscal pressures, 
fragmented supply chains, and an increased reliance on economic policy 
instruments for strategic purposes.
Against this backdro


In [4]:
print(pages[2].page_content[:500])

State of the Economy
33
within and across countries. Growth in the US has remained strong, primarily driven 
by investment in artificial intelligence (AI). Total IT investment, which also includes 
spending by businesses on equipment and software to facilitate AI use, has accounted for 
nearly half of GDP growth in recent quarters, helping to mitigate the negative effects of 
trade tariffs on growth.1 This strong growth has been accompanied by inflation remaining 
stubbornly above the 2 per cent


In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,  # ~150 words per chunk
    chunk_overlap=100,  # overlap keeps context at boundaries
    separators=["\n\n", "\n", ".", " "],  # tries paragraph → line → sentence → word
)

chunks = splitter.split_documents(pages)

print(f"Total chunks: {len(chunks)}  (from {len(pages)} pages)")
print(
    f"Avg chunk length: {sum(len(c.page_content) for c in chunks) // len(chunks)} chars"
)
print("\n--- Example chunk ---")
print(chunks[5].page_content)

Total chunks: 193  (from 35 pages)
Avg chunk length: 521 chars

--- Example chunk ---
Economic Survey 2025-26
22
GLOBAL ECONOMIC GROWTH – FRAGILE AND DIVERGING
1.1. Since the last version of the Economic Survey was published, the global economy has 
been subjected to multiple upheavals. The most disruptive amongst these disturbances 
was the imposition of tariffs by the USA on imports from its trade partners. The long 
promised reciprocal tariffs, announced in April 2025, initially sparked concerns about 
lower growth and higher inflation in the global economy which have proven to be 
transient in the short run. This was due to multiple reasons. Trade agreements between


In [6]:
print(chunks[1].page_content)

Against this backdrop, the Indian economy has maintained strong growth 
momentum in FY26. The First Advance Estimates place real GDP growth at 7.4 
per cent, with growth largely driven by domestic demand. Private consumption 
and capital formation continue to support expansion, while services remain the 
key contributor on the supply side. Manufacturing activity has strengthened, 
and agriculture has provided stability, notwithstanding structural constraints.
The chapter analyses demand and supply-side developments during the first


In [7]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

print("Loading embedding model (downloads ~90 MB on first run)...")
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

print("Embedding all chunks and storing in Chroma (in memory)...")
vector_store = Chroma.from_documents(chunks, embeddings)

print(f"Vector store ready. {vector_store._collection.count()} vectors stored.")

Loading embedding model (downloads ~90 MB on first run)...


Loading weights: 100%|██████████| 103/103 [00:02<00:00, 47.62it/s]


Embedding all chunks and storing in Chroma (in memory)...
Vector store ready. 193 vectors stored.


In [8]:
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

test_query = "What is VoLTE and how does it improve call quality?"
retrieved = retriever.invoke(test_query)

print(f"Query: {test_query}")
print(f"Retrieved {len(retrieved)} chunks:\n")
for i, doc in enumerate(retrieved, 1):
    print(f"--- Chunk {i} ---")
    print(doc.page_content[:300])
    print()

Query: What is VoLTE and how does it improve call quality?
Retrieved 3 chunks:

--- Chunk 1 ---
annual surveys such as the comprehensive modular surveys on telecom and education in 
2025.
Modernising surveys and strengthening state statistical systems 
MoSPI has undertaken major survey system reforms and survey modernisation initiatives 
aimed at increasing both the frequency and diversity of 

--- Chunk 2 ---
services related to
broadcasting
   Financial,  real estate
&  professional  services
   Public administration,
defence  and Other
Services
YoY Growth at constant prices
H1:FY16-FY20 H1:FY25 H1:FY26
Source: MoSPI

--- Chunk 3 ---
1.19.  Regarding the industry, a concern is often raised about its declining share in 
GVA. The compression in manufacturing’s GVA share stems from relative price effects 
rather than reflecting a decline in manufacturing activities (See section ‘GDP Deflators: 
Manufacturing’s Reversal in Terms of 



In [9]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_groq import ChatGroq


# --- Helper: join retrieved chunks into a single context string ---
def format_docs(docs):
    return "\n\n---\n\n".join(doc.page_content for doc in docs)


# --- System prompt: ground the LLM in the retrieved context ---
SYSTEM_PROMPT = """\
You are a PUSHING THE GROWTH
FRONTIER assistant.
Answer the question using ONLY the context provided below.
If the context does not contain enough information, say so clearly.

Context:
{context}
"""

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", SYSTEM_PROMPT),
        ("human", "{question}"),
    ]
)

# --- LLM via Groq API ---
llm = ChatGroq(
    model="qwen/qwen3-32b",
    temperature=0,
    reasoning_format="parsed",
)

# --- Assemble the chain ---
chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("RAG chain assembled.")

RAG chain assembled.


## Step 6 — Ask a Single Question

In [10]:
question = "How does GDP should increase"

print(f"Q: {question}\n")
print("A:", chain.invoke(question))

Q: How does GDP should increase

A: Based on the context provided, GDP growth can be enhanced through the following mechanisms:

1. **Investment in Technology and AI**: The context highlights that investment in artificial intelligence (AI) and related IT infrastructure accounts for nearly half of recent GDP growth in the U.S., mitigating trade tariff impacts. Prioritizing such investments can drive productivity and economic expansion.

2. **Improving Total Factor Productivity (TFP)**: TFP growth, derived from efficiency gains in capital (K), labor (L), and output (Y), is critical. The context notes that sustained reforms and smoothing disruptions (e.g., post-pandemic adjustments) can elevate trend TFP, contributing to higher potential GDP growth.

3. **Strengthening Capital and Labor Inputs**: The context mentions that calibrated improvements in capital accumulation and labor input, combined with TFP growth, can shift potential GDP growth upward (e.g., from 6.5% to 7% medium-term). Pol